# MarketGraph + THGNN Native Adapter v1

目标：把 `market_graph_mse` 和 `thgnn_mse` 从 feature-proxy 推进到 `src/model` 的 native graph estimator，并用同一套 `src/analysis` / qlib-style regime-window 框架记录区间和 regime 下的模型差异。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.graph import MarketGraphRegressor, THGNNGraphRegressor
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'thgnn_market_graph_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'
MODEL_NAMES = ('ridge_mse', 'market_graph_mse', 'hypergraph_mse', 'thgnn_mse', 'sthan_mse', 'sthgcn_mse')

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=MODEL_NAMES,
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:8])

market_estimator, market_constant = fit_estimator_for_model('market_graph_mse', train=train, feature_columns=feature_columns)
thgnn_estimator, thgnn_constant = fit_estimator_for_model('thgnn_mse', train=train, feature_columns=feature_columns)
capabilities = model_capability_table(('market_graph_mse', 'thgnn_mse')).set_index('model')
fidelity = model_fidelity_summary()

assert market_constant is None
assert thgnn_constant is None
assert isinstance(market_estimator, MarketGraphRegressor)
assert isinstance(thgnn_estimator, THGNNGraphRegressor)
assert market_estimator.feature_count_ == 8
assert market_estimator.node_count_ == 8
assert market_estimator.market_adjacency_.shape == (8, 8)
assert market_estimator.market_context_gate_.shape == (8,)
assert thgnn_estimator.feature_count_ == 8
assert thgnn_estimator.node_count_ == 8
assert thgnn_estimator.temporal_channel_count_ == 4
assert thgnn_estimator.relation_count_ >= 4
assert thgnn_estimator.relation_adjacencies_.shape == (thgnn_estimator.relation_count_, 8, 8)
assert thgnn_estimator.temporal_attention_weights_.shape == (4,)
assert thgnn_estimator.relation_attention_weights_.shape == (thgnn_estimator.relation_count_,)
assert capabilities.loc['market_graph_mse', 'paper_fidelity'] == 'native_graph'
assert capabilities.loc['market_graph_mse', 'is_proxy'] is False
assert capabilities.loc['thgnn_mse', 'paper_fidelity'] == 'native_graph'
assert capabilities.loc['thgnn_mse', 'is_proxy'] is False
assert model_adapter_name('market_graph_mse') == 'market_structure_graph_mse_v1'
assert model_adapter_name('thgnn_mse') == 'thgnn_temporal_hypergraph_mse_v1'
assert len(fidelity['native_models']) == 28
assert len(fidelity['proxy_models']) == 4

comparison = outputs.model_comparison.set_index('model')
ledger = outputs.model_run_ledger.set_index('model')
summary = {
    'schema_version': 'research.thgnn_market_graph_native_adapter_v1',
    'status': 'completed',
    'models': ['market_graph_mse', 'thgnn_mse'],
    'adapters': {
        'market_graph_mse': model_adapter_name('market_graph_mse'),
        'thgnn_mse': model_adapter_name('thgnn_mse'),
    },
    'estimators': {
        'market_graph_mse': market_estimator.__class__.__name__,
        'thgnn_mse': thgnn_estimator.__class__.__name__,
    },
    'market_graph': {
        'node_count': market_estimator.node_count_,
        'feature_count': market_estimator.feature_count_,
        'hidden_size': market_estimator.hidden_size,
        'market_adjacency_shape': list(market_estimator.market_adjacency_.shape),
        'market_context_gate_shape': list(market_estimator.market_context_gate_.shape),
    },
    'thgnn': {
        'node_count': thgnn_estimator.node_count_,
        'feature_count': thgnn_estimator.feature_count_,
        'temporal_channel_count': thgnn_estimator.temporal_channel_count_,
        'relation_count': thgnn_estimator.relation_count_,
        'hidden_size': thgnn_estimator.hidden_size,
        'relation_adjacencies_shape': list(thgnn_estimator.relation_adjacencies_.shape),
    },
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'model_fidelity_counts': fidelity['fidelity_counts'],
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'matrix_models': sorted(outputs.predictions['model'].astype(str).unique().tolist()),
    'target_metrics': {
        model: {
            'return_pct': float(comparison.loc[model, 'return_pct']),
            'relative_return_pct': float(comparison.loc[model, 'relative_return_pct']),
            'sharpe': float(comparison.loc[model, 'sharpe']),
            'max_drawdown_pct': float(comparison.loc[model, 'max_drawdown_pct']),
            'stability_score': float(comparison.loc[model, 'stability_score']),
            'trained': bool(ledger.loc[model, 'trained']),
            'backtested': bool(ledger.loc[model, 'backtested']),
        }
        for model in ['market_graph_mse', 'thgnn_mse']
    },
    'analysis_layers': outputs.summary.get('analysis_layers', []),
    'analysis_table_counts': outputs.summary.get('analysis_table_counts', {}),
    'external_references': [
        {
            'source_id': 'arxiv:1809.09441',
            'url': 'https://arxiv.org/abs/1809.09441',
            'usage': '参考关系股票排序/图关系建模方向；market_graph_mse 仅实现 compact market-structure feature graph，不声称复现完整论文。',
        },
        {
            'source_id': 'doi:10.1016/j.asoc.2024.111329',
            'url': 'https://doi.org/10.1016/j.asoc.2024.111329',
            'usage': '参考动态 hypergraph spatio-temporal stock trend prediction 的结构方向；本实现只保留 compact adapter。',
        },
        {
            'source_id': 'github:tanjingme/CI-STHPAN',
            'url': 'https://github.com/tanjingme/CI-STHPAN',
            'usage': '参考开源 spatio-temporal hypergraph attention stock selection 的边界；未复制第三方代码。',
        },
    ],
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
        'qlib_risk_metrics': str((CARD_DIR / 'qlib_risk_metrics.csv').relative_to(REPO_ROOT)),
        'model_effect_deltas': str((CARD_DIR / 'model_effect_deltas.csv').relative_to(REPO_ROOT)),
        'model_regime_window_diagnostics': str((CARD_DIR / 'model_regime_window_diagnostics.csv').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.thgnn_market_graph_native_adapter_v1',
 'status': 'completed',
 'models': ['market_graph_mse', 'thgnn_mse'],
 'adapters': {'market_graph_mse': 'market_structure_graph_mse_v1',
  'thgnn_mse': 'thgnn_temporal_hypergraph_mse_v1'},
 'estimators': {'market_graph_mse': 'MarketGraphRegressor',
  'thgnn_mse': 'THGNNGraphRegressor'},
 'market_graph': {'node_count': 8,
  'feature_count': 8,
  'hidden_size': 8,
  'market_adjacency_shape': [8, 8],
  'market_context_gate_shape': [8]},
 'thgnn': {'node_count': 8,
  'feature_count': 8,
  'temporal_channel_count': 4,
  'relation_count': 5,
  'hidden_size': 8,
  'relation_adjacencies_shape': [5, 8, 8]},
 'full_matrix_native_model_count': 28,
 'full_matrix_proxy_model_count': 4,
 'model_fidelity_counts': {'native_graph': 8,
  'native_library': 8,
  'native_sequence': 6,
  'native_torch': 6,
  'proxy_feature_adapter': 4},
 'matrix_card': 'research/thgnn_market_graph_native_adapter_v1/matrix_card',
 'matrix_models': ['hypergra

## 外部参考摘要

- `Temporal Relational Ranking for Stock Prediction`：关系股票预测/排序方向可作为 market graph 的思想参考。参考：https://arxiv.org/abs/1809.09441
- `Stock trend prediction based on dynamic hypergraph spatio-temporal network`：公开题录体现动态 hypergraph 与时空趋势预测结合。参考：https://doi.org/10.1016/j.asoc.2024.111329
- `CI-STHPAN`：开源项目聚焦 channel-independent spatio-temporal hypergraph attention stock selection。参考：https://github.com/tanjingme/CI-STHPAN
- 本仓库实现边界：SOXL 单资产监督任务当前没有真实股票池横截面、行业图或 ETF 成分关系，因此本轮只把这些思想落成 compact native adapter；后续接入多资产/真实关系图时，应替换 graph provider，而不是把关系写死在 estimator。

## 中文结论

- 做了什么：在 `src/model.graph` 保留框架内 native graph estimator，`market_graph_mse` 使用市场强度 gate + feature graph propagation，`thgnn_mse` 使用四通道 temporal signal、训练样本相关性/chain/hypergraph 多关系邻接和 temporal/relation attention。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`market_graph_mse`、`hypergraph_mse`、`thgnn_mse`、`sthan_mse`、`sthgcn_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=28、proxy=4；`market_graph_mse` 和 `thgnn_mse` 均为 `native_graph`，不再计入 proxy。
- Regime/区间差异：本轮继续复用 `src/analysis` 的 window/regime/qlib 风险表，具体强弱分段以 `matrix_card/model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 `model_review_briefing.csv` 为准。
- regime 差异：本 notebook 不额外引入新的 regime 结论；如有分层结果，以已写出的 analysis/regime/window sidecar 为准。
- 是否晋升：否。本次是模型实现和分析链路覆盖升级，不直接晋升生产策略。
- 风险：当前 graph node 仍是 SOXL 支撑特征列，不是真实股票关系图；THGNN-style compact adapter 不能等价完整多资产论文复现，真实横截面股票池、行业/ETF 成分、成交/资金流关系和更长 walk-forward 仍需验证。
- 下一步：继续补齐剩余 proxy：`ridge_ic`、`extra_trees_rank`、`sequence_lag_ridge_mse`、`sthgcn_mse`；其中 `sthgcn_mse` 应优先沿 graph/native 路线推进。